# Utility notebook to translate species-specific genes from old gene names to new

This is something to manually run if you regenerate the prodigal gene calls and/or modify the
cds extraction code in `workflows/mapping.smk`.

BLASTN columns (`--output-fmt 6`)

```
   1.  qseqid      query (gene) sequence id
   2.  sseqid      subject (reference genome) sequence id
   3.  pident      percentage of identical positions
   4.  length      alignment length (sequence overlap)
   5.  mismatch    number of mismatches
   6.  gapopen     number of gap openings
   7.  qstart      start of alignment in query
   8.  qend        end of alignment in query
   9.  sstart      start of alignment in subject
 10.  send        end of alignment in subject
 11.  evalue      expect value
 12.  bitscore    bit score
```

In [1]:
import polars as pl

In [2]:
df = pl.read_csv('../2025-workflow-genes/new/old-x-new.blastn',
                 separator='\t', has_header=False,
                 new_columns=['qseqid', 'sseqid', 'pident', 'length', 'mismatch', 'gapopen', 'qstart', 'qend', 'sstart', 'ssend', 'evalue', 'bitscore'])

In [3]:
df

qseqid,sseqid,pident,length,mismatch,gapopen,qstart,qend,sstart,ssend,evalue,bitscore
str,str,f64,i64,i64,i64,i64,i64,i64,i64,f64,f64
"""FGLPJJIK_03110""","""AtH2023_SRR7182046_MAG5_337_5""",95.273,825,38,1,1,825,82,905,0.0,1306.0
"""NLNOELNI_01902""","""AtH2023_ERR3211826_MAG3_378_5""",99.207,1766,14,0,278,2043,305,2070,0.0,3184.0
"""MMBDDDLD_00384""","""AtH2023_SRR8960082_MAG05_330_3""",95.892,998,35,3,1,995,1,995,0.0,1611.0
"""MMBDDDLD_00384""","""AtH2023_SRR5976187_MAG13_87_1""",86.852,791,99,4,247,1035,206,993,0.0,880.0
"""MMBDDDLD_00384""","""AtH2023_ERR1135458_MAG11_234_5""",92.346,601,44,2,1,600,1,600,0.0,854.0
…,…,…,…,…,…,…,…,…,…,…,…
"""AKLOHKAG_01416""","""AtH2023_SRR17241623_MAG06_3_11""",85.67,321,42,4,1,319,10,328,6.6900e-92,335.0
"""AKLOHKAG_01416""","""AtH2023_SRR17241623_MAG06_3_11""",88.446,251,25,4,424,673,3,250,2.4400e-81,300.0
"""AKLOHKAG_01416""","""AtH2023_SRR17241623_MAG06_3_11""",88.341,223,24,2,1,222,96,317,2.4800e-71,267.0


## Get highest-bitscore matches between query and subject sequence IDs

In [4]:
sum_df = (df.sort('bitscore', descending=True).group_by(['qseqid'], maintain_order=True).agg(
        pl.col('sseqid'),
        pl.col('bitscore'),
    ).with_columns(
        sseqid=pl.col('sseqid').list.get(0),
        bitscore=pl.col('bitscore').list.get(0))
    .filter(pl.col('bitscore') > 600)
)

In [5]:
sum_df

qseqid,sseqid,bitscore
str,str,f64
"""APOLKBKG_00160""","""AtH2023_SRR11489784_MAG01_15_2""",6006.0
"""FCHBMNJF_01297""","""AtH2023_SRR14369225_MAG28_11_7…",5762.0
"""BIBFGMOK_00552""","""AtH2023_SRR17241510_MAG40_8_10""",4972.0
"""OBGMJDLM_01221""","""AtH2023_SRR17241544_MAG06_60_1""",4723.0
"""GFALEAHI_01277""","""AtH2023_ERR3211937_MAG3_16_9""",4320.0
…,…,…
"""IEPMJGAI_00771""","""AtH2023_SRR8960673_MAG08_57_36""",915.0
"""LJHBCGLM_01294""","""AtH2023_SRR17241641_MAG31_88_1""",865.0
"""NADIOELI_00170""","""AtH2023_SRR17241638_MAG02_2_35""",865.0


In [6]:
species_genes_df = pl.read_csv('../2025-workflow-genes/species-genes.csv')
species_genes_df

good,anchor,species,gene_name,description
i64,i64,str,str,str
1,1,"""s__Phascolarctobacterium_A suc…","""CNENGHLA_01260""","""BLAST match to hydrogenase lar…"
1,0,"""s__Phascolarctobacterium_A suc…","""EHOAPHDI_01174""","""BLAST match to protein phospha…"
1,0,"""s__Phascolarctobacterium_A suc…","""CNENGHLA_00658""","""BLAST match to 4-hydroxy-3-met…"
1,0,"""s__Phascolarctobacterium_A suc…","""IFIBFMPA_00800""","""BLAST match to 2-isopropylmal…"
1,0,"""s__Lactobacillus amylovorus""","""BBOFCOCJ_01349""","""BLAST match to peptidase T [La…"
…,…,…,…,…
0,0,"""s__Cryptobacteroides sp0004349…","""ADCODNMF_01281""","""(not reviewed)"""
0,0,"""s__Cryptobacteroides sp0004349…","""EIGGHFPC_02121""","""(not reviewed)"""
0,0,"""s__Cryptobacteroides sp0004349…","""DOLCGLJK_00897""","""(not reviewed)"""


In [7]:
new_genes_df = species_genes_df.join(sum_df.drop('bitscore'),
                                     left_on='gene_name',
                                     right_on='qseqid', how='inner')

In [8]:
new_genes_df  = new_genes_df.drop('gene_name').rename({'sseqid': 'gene_name'})

In [9]:
new_genes_df.write_csv('../inputs.genes/TRY-species-genes.csv')

# rename to / concat with 'species-genes.csv'.

In [10]:
new_genes_df

good,anchor,species,description,gene_name
i64,i64,str,str,str
1,0,"""s__Lactobacillus amylovorus""","""BLAST match to peptidase T [La…","""AtH2023_SRR11125358_MAG1_217_2…"
1,0,"""s__Lactobacillus amylovorus""","""BLAST match to calcium-translo…","""AtH2023_SRR17241544_MAG06_60_1"""
1,1,"""s__Lactobacillus amylovorus""","""BLAST match to LD-transpeptida…","""AtH2023_SRR8960431_MAG03_43_3"""
1,0,"""s__Bariatricus sp004560705""","""BLAST match to competence prot…","""AtH2023_SRR17241619_MAG37_18_8"""
1,0,"""s__Mogibacterium_A kristiansen…","""BLAST match to isolate genome""","""AtH2023_SRR17241623_MAG06_3_11"""
…,…,…,…,…
1,1,"""s__Cryptobacteroides sp0004349…","""hypothetical protein [Bacteroi…","""AtH2023_SRR8960403_MAG06_120_8"""
1,0,"""s__Cryptobacteroides sp0004349…","""putative uncharacterized prote…","""AtH2023_SRR17241635_MAG12_71_7"""
1,0,"""s__Cryptobacteroides sp0004349…","""ATP-binding protein [Bacteroid…","""AtH2023_SRR14369225_MAG28_11_7…"
